<a href="https://colab.research.google.com/github/kaixing189/MSSP607/blob/main/Week_05_Participation_Activity.ipynbb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MSSP6070 Week 5 Participation Activity

## Working with Multiple Stock Data Files in Pandas

本 Notebook 完成 Week 5 Participation Activity 的全部数据处理任务。

**本次要完成的内容：**

1. 从 Colab Secrets 读取 `Git_Key`。
2. Clone 自己的 GitHub repository 到 Google Colab。
3. 找到 `aapl.csv`、`amzn.csv`、`fb.csv`、`nflx.csv` 和 `goog.csv`。
4. 分别读取五个 CSV 文件。
5. 给每个 DataFrame 添加 `ticker` 列。
6. 将五个 DataFrame 合并成一个 DataFrame。
7. 保存为 `faanf.csv`。
8. 将 `date` 转为 datetime，将 `volume` 转为 integer，并按 `date`、`ticker` 排序。
9. 找出 volume 最大的 7 行。
10. 使用 `melt()` 将数据从 wide format 转成长格式。
11. 检查最终文件，并完成 GitHub 和 Canvas 提交。

> **注意：** 作业第 2(e) 要求保存为 `faanf.csv`，但第 2(f) 写的是 `fanng.csv`。这两个名称不一致。为了兼容作业文字，本 Notebook 会保存 `faanf.csv`，并同时生成一个内容相同的 `fanng.csv`。后面的 2(f) 会读取 `fanng.csv`，这样严格按照题目文字也能运行。


## Part 0. 第一次使用前需要做什么

在运行 Notebook 之前，请确认：

1. 你已经在 Colab 左侧 **Secrets** 中创建了名为 `Git_Key` 的 Secret。
2. `Git_Key` 的 Notebook access 开关已经打开。
3. 你的 GitHub Personal Access Token 对 MSSP6070 repository 至少有 **Contents: Read and write** 权限。
4. 你已经把五个原始 CSV 文件放进自己的 GitHub repository。

推荐结构：

```text
MSSP6070/
├── data/
│   └── Week05/
│       ├── aapl.csv
│       ├── amzn.csv
│       ├── fb.csv
│       ├── goog.csv
│       └── nflx.csv
└── WeeklyModules/
    └── Week05/
        └── Week_05_Participation_Activity.ipynb
```

如果你的五个 CSV 在 repository 的根 `data/` 文件夹或 `WeeklyModules/Week05/data/` 中也没有关系。下面的代码会自动检查几个常见位置。


## Part 1. 连接 GitHub

### 这一部分在做什么？

这一部分从 Colab Secrets 中读取 `Git_Key`，然后把你的 GitHub repository clone 到 Colab 的临时运行环境中。

你只需要修改下面的 **`REPO_URL`**。把它替换成你自己的 repository HTTPS URL，例如：

```text
https://github.com/your_username/MSSP6070.git
```

**不要把真实 GitHub Token 写进 Notebook，也不要 print Token。**


In [1]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess

# ============================================================
# 只需要修改这一行：粘贴你自己的 GitHub repository HTTPS URL
# ============================================================
REPO_URL = "https://github.com/kaixing189/MSSP607.git"

# 从 Colab Secrets 读取 Git_Key
github_token = userdata.get('Git_Key')

if github_token is None:
    raise ValueError(
        "找不到 Git_Key。请检查 Colab Secrets 中的名称，"
        "并确认当前 Notebook 的 access 开关已经打开。"
    )

if "YOUR_USERNAME" in REPO_URL or not REPO_URL.startswith("https://github.com/"):
    raise ValueError(
        "请先把 REPO_URL 改成你自己的 GitHub repository HTTPS URL。"
    )

print("Git_Key 已成功读取。Token 内容不会显示。")


Git_Key 已成功读取。Token 内容不会显示。


### 安全地 clone repository

下面使用 `GIT_ASKPASS` 临时向 GitHub 提供 Token。这样可以避免把 Token 直接写进 repository URL。

如果当前 Runtime 已经 clone 过这个 repository，代码不会重复 clone，而是直接进入已有文件夹。


In [2]:
# 从 URL 中得到 repository 名称，例如 MSSP6070
repo_name = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
repo_dir = Path('/content') / repo_name

# 创建一个临时的 Git askpass 脚本。
# Token 不会写进这个脚本，只通过环境变量临时传入。
askpass_path = Path('/tmp/git_askpass.sh')
askpass_path.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
    'esac\n'
)
askpass_path.chmod(0o700)

git_env = os.environ.copy()
git_env['GIT_ASKPASS'] = str(askpass_path)
git_env['GIT_TERMINAL_PROMPT'] = '0'
git_env['GITHUB_TOKEN'] = github_token

if not repo_dir.exists():
    subprocess.run(
        ['git', 'clone', REPO_URL, str(repo_dir)],
        env=git_env,
        check=True
    )
    print(f"Repository 已 clone 到: {repo_dir}")
else:
    print(f"Repository 已经存在于本次 Runtime: {repo_dir}")

os.chdir(repo_dir)
print("当前工作文件夹:", os.getcwd())


Repository 已 clone 到: /content/MSSP607
当前工作文件夹: /content/MSSP607


## Part 2. 找到 Week 5 的五个 CSV 文件

### 这一部分在做什么？

不同同学可能使用不同的 GitHub 文件夹结构，所以这里会自动检查几个常见的位置。

代码要找到这五个文件：

* `aapl.csv`
* `amzn.csv`
* `fb.csv`
* `goog.csv`
* `nflx.csv`

如果找不到，代码会告诉你应该检查 GitHub 中的数据文件位置。


In [9]:
%cd /content/MSSP607

!git pull

/content
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 9 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 47.42 KiB | 758.00 KiB/s, done.
From https://github.com/kaixing189/MSSP607
   62654da..825dbe4  main       -> origin/main
Updating 62654da..825dbe4
Fast-forward
 data/aapl.csv  |  252 ++++++++++++
 data/amzn.csv  |  252 ++++++++++++
 data/faanf.csv | 1256 ++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 data/faang.csv | 1256 ++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 data/fanng.csv | 1256 ++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 data/fb.csv    |  252 ++++++++++++
 data/goog.csv  |  252 ++++++++++++
 data/nflx.csv  |  252 ++++++++++++
 8 files changed, 5028 insertions(+)
 create mode 100644 data/aapl.csv
 create mode 100644 data/amzn.csv
 create mode 100644 data/faanf.csv
 create mode 100644 data/faang.csv
 c

In [10]:
required_files = [
    'aapl.csv',
    'amzn.csv',
    'fb.csv',
    'goog.csv',
    'nflx.csv'
]

candidate_dirs = [
    repo_dir / 'data' / 'Week05',
    repo_dir / 'data' / 'Week 05 Examples',
    repo_dir / 'WeeklyModules' / 'Week05' / 'data',
    repo_dir / 'data'
]

DATA_DIR = None

for folder in candidate_dirs:
    if folder.exists() and all((folder / f).exists() for f in required_files):
        DATA_DIR = folder
        break

if DATA_DIR is None:
    print("没有在常见位置找到全部 5 个 CSV。")
    print("请确认以下文件已经上传到 GitHub repository 的 data 文件夹：")
    for f in required_files:
        print("  -", f)
    print("\nRepository 当前文件结构中找到的 CSV：")
    for p in repo_dir.rglob('*.csv'):
        print(" ", p.relative_to(repo_dir))
    raise FileNotFoundError("未找到 Week 5 所需的全部 CSV 文件。")

print("使用的数据文件夹:", DATA_DIR)
print("\n检查文件:")
for f in required_files:
    print(f"{f}: {(DATA_DIR / f).exists()}")


使用的数据文件夹: /content/MSSP607/data

检查文件:
aapl.csv: True
amzn.csv: True
fb.csv: True
goog.csv: True
nflx.csv: True


## Part 3. 导入 Pandas，并分别读取五个文件

### 对应题目 2(b)

这一部分使用 `pd.read_csv()` 将每个 CSV 文件读取为独立的 Pandas DataFrame。

读取后会显示每个 DataFrame 的行数和列数。原始 Week 5 文件中，每只股票应该有 **251 行 × 6 列**。


In [11]:
import pandas as pd

aapl = pd.read_csv(DATA_DIR / 'aapl.csv')
amzn = pd.read_csv(DATA_DIR / 'amzn.csv')
fb   = pd.read_csv(DATA_DIR / 'fb.csv')
goog = pd.read_csv(DATA_DIR / 'goog.csv')
nflx = pd.read_csv(DATA_DIR / 'nflx.csv')

print('AAPL:', aapl.shape)
print('AMZN:', amzn.shape)
print('FB:  ', fb.shape)
print('GOOG:', goog.shape)
print('NFLX:', nflx.shape)

aapl.head()


AAPL: (251, 6)
AMZN: (251, 6)
FB:   (251, 6)
GOOG: (251, 6)
NFLX: (251, 6)


,date,open,high,low,close,volume
0,2018-01-02,166.9271,169.0264,166.0442,168.9872,25555934
1,2018-01-03,169.2521,171.2337,168.6929,168.9578,29517899
2,2018-01-04,169.2619,170.1742,168.8106,169.7426,22434597
3,2018-01-05,170.1448,172.0381,169.7622,171.6751,23660018
4,2018-01-08,171.0375,172.2736,170.6255,171.0375,20567766


## Part 4. 给每个 DataFrame 添加 `ticker` 列

### 对应题目 2(c)

五个 CSV 原本只有日期、价格和成交量，因此合并之后如果没有额外标识，我们无法知道某一行属于哪只股票。

所以这里新增 `ticker` 列：

* Apple → `AAPL`
* Amazon → `AMZN`
* Facebook → `FB`
* Google → `GOOG`
* Netflix → `NFLX`


In [12]:
aapl['ticker'] = 'AAPL'
amzn['ticker'] = 'AMZN'
fb['ticker']   = 'FB'
goog['ticker'] = 'GOOG'
nflx['ticker'] = 'NFLX'

# 查看一个例子，确认 ticker 已经成功加入
aapl.head()


,date,open,high,low,close,volume,ticker
0,2018-01-02,166.9271,169.0264,166.0442,168.9872,25555934,AAPL
1,2018-01-03,169.2521,171.2337,168.6929,168.9578,29517899,AAPL
2,2018-01-04,169.2619,170.1742,168.8106,169.7426,22434597,AAPL
3,2018-01-05,170.1448,172.0381,169.7622,171.6751,23660018,AAPL
4,2018-01-08,171.0375,172.2736,170.6255,171.0375,20567766,AAPL


## Part 5. 将五个 DataFrame 合并

### 对应题目 2(d)

这里使用 `pd.concat()` 把五只股票的数据**纵向追加**到同一个 DataFrame。

`ignore_index=True` 会重新生成连续的行号。

五个文件各有 251 行，因此合并后应该有：

**251 × 5 = 1,255 rows**


In [13]:
faanf = pd.concat(
    [aapl, amzn, fb, nflx, goog],
    ignore_index=True
)

print('Combined shape:', faanf.shape)
print('\nColumns:')
print(faanf.columns.tolist())

faanf.head()


Combined shape: (1255, 7)

Columns:
['date', 'open', 'high', 'low', 'close', 'volume', 'ticker']


,date,open,high,low,close,volume,ticker
0,2018-01-02,166.9271,169.0264,166.0442,168.9872,25555934,AAPL
1,2018-01-03,169.2521,171.2337,168.6929,168.9578,29517899,AAPL
2,2018-01-04,169.2619,170.1742,168.8106,169.7426,22434597,AAPL
3,2018-01-05,170.1448,172.0381,169.7622,171.6751,23660018,AAPL
4,2018-01-08,171.0375,172.2736,170.6255,171.0375,20567766,AAPL


**预期结果：** `faanf` 应该有 **1,255 行 × 7 列**。

7 列分别是：`date`, `open`, `high`, `low`, `close`, `volume`, `ticker`。


## Part 6. 保存合并结果为 `faanf.csv`

### 对应题目 2(e)

这里把合并后的 DataFrame 写回 GitHub repository 的 Week 5 data 文件夹。

`index=False` 表示不要把 Pandas 自动生成的行号保存成额外的一列。

因为下一题写的是 `fanng.csv`，本 Notebook 也会保存一个内容完全相同的 `fanng.csv` 兼容文件。


In [14]:
faanf_path = DATA_DIR / 'faanf.csv'
fanng_path = DATA_DIR / 'fanng.csv'

faanf.to_csv(faanf_path, index=False)

# 兼容题目第 2(f) 使用的文件名
faanf.to_csv(fanng_path, index=False)

print('Saved:', faanf_path)
print('Compatibility copy saved:', fanng_path)


Saved: /content/MSSP607/data/faanf.csv
Compatibility copy saved: /content/MSSP607/data/fanng.csv


## Part 7. 类型转换并按 `date` 和 `ticker` 排序

### 对应题目 2(f)

这一部分重新读取题目写出的 `fanng.csv`，然后进行两种类型转换：

* `date` → `datetime`
* `volume` → integer

最后先按 `date` 排序，同一天内再按 `ticker` 排序。


In [15]:
fanng = pd.read_csv(fanng_path)

# date: object/string -> datetime
fanng['date'] = pd.to_datetime(fanng['date'])

# volume -> integer
fanng['volume'] = fanng['volume'].astype('int64')

# 按日期和 ticker 排序
fanng = fanng.sort_values(
    by=['date', 'ticker']
).reset_index(drop=True)

print(fanng.dtypes)
fanng.head(10)


date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume             int64
ticker            object
dtype: object


,date,open,high,low,close,volume,ticker
0,2018-01-02,166.9271,169.0264,166.0442,168.9872,25555934,AAPL
1,2018-01-02,1172.0000,1190.0000,1170.5100,1189.0100,2694494,AMZN
2,2018-01-02,177.6800,181.5800,177.5500,181.4200,18151903,FB
3,2018-01-02,1048.3400,1066.9400,1045.2300,1065.0000,1237564,GOOG
4,2018-01-02,196.1000,201.6500,195.4200,201.0700,10966889,NFLX
5,2018-01-03,169.2521,171.2337,168.6929,168.9578,29517899,AAPL
6,2018-01-03,1188.3000,1205.4900,1188.3000,1204.2000,3108793,AMZN
7,2018-01-03,181.8800,184.7800,181.3300,184.6700,16886563,FB
8,2018-01-03,1064.3100,1086.2900,1063.2100,1082.4800,1430170,GOOG
9,2018-01-03,202.0500,206.2100,201.5000,205.0500,8591369,NFLX


### 检查类型转换

运行上面的 cell 后：

* `date` 应该显示为 `datetime64[ns]`
* `volume` 应该显示为 `int64`


## Part 8. 找出 volume 最大的 7 行

### 对应题目 2(g)

`nlargest(7, 'volume')` 会直接找到 `volume` 数值最大的 7 条记录。

我们显示日期、ticker 和 volume，方便直接检查答案。


In [16]:
top_7_volume = fanng.nlargest(7, 'volume')

top_7_volume[['date', 'ticker', 'volume']]


,date,ticker,volume
712,2018-07-26,FB,169803668
267,2018-03-20,FB,129851768
287,2018-03-26,FB,126116634
272,2018-03-21,FB,106598834
910,2018-09-21,AAPL,96246748
1225,2018-12-21,AAPL,95744384
1060,2018-11-02,AAPL,91328654


### 使用老师提供的 Week 5 原始文件时，预期的 7 行是：

| Date | Ticker | Volume |
|---|---|---:|
| 2018-07-26 | FB | 169,803,668 |
| 2018-03-20 | FB | 129,851,768 |
| 2018-03-26 | FB | 126,116,634 |
| 2018-03-21 | FB | 106,598,834 |
| 2018-09-21 | AAPL | 96,246,748 |
| 2018-12-21 | AAPL | 95,744,384 |
| 2018-11-02 | AAPL | 91,328,654 |


## Part 9. 使用 `melt()` 将数据转换成长格式

### 对应题目 2(h)

原来的数据是 **wide format**：

```text
date | ticker | open | high | low | close | volume
```

题目要求把 `date` 和 `ticker` 保留为 key identifiers，并把其他列 melt 到一起。

转换以后会变成 **long format**：

```text
date | ticker | variable | value
```

`variable` 会包含 `open`, `high`, `low`, `close`, `volume`。


In [17]:
melted = fanng.melt(
    id_vars=['date', 'ticker'],
    value_vars=['open', 'high', 'low', 'close', 'volume'],
    var_name='variable',
    value_name='value'
)

print('Original shape:', fanng.shape)
print('Melted shape:  ', melted.shape)

melted.head(15)


Original shape: (1255, 7)
Melted shape:   (6275, 4)


,date,ticker,variable,value
0,2018-01-02,AAPL,open,166.9271
1,2018-01-02,AMZN,open,1172.0000
2,2018-01-02,FB,open,177.6800
3,2018-01-02,GOOG,open,1048.3400
4,2018-01-02,NFLX,open,196.1000
5,2018-01-03,AAPL,open,169.2521
6,2018-01-03,AMZN,open,1188.3000
7,2018-01-03,FB,open,181.8800
8,2018-01-03,GOOG,open,1064.3100
9,2018-01-03,NFLX,open,202.0500


**预期结果：**

* 原数据：**1,255 × 7**
* melt 后：**6,275 × 4**

因为每一条股票记录原来有 5 个需要 melt 的 measurement columns，所以：

`1,255 × 5 = 6,275 rows`


## Part 10. 可选：保存排序后的数据和 melted 数据

题目只明确要求保存 `faanf.csv`，但保存这两个结果可以方便你以后检查。


In [18]:
sorted_path = DATA_DIR / 'faanf_sorted.csv'
melted_path = DATA_DIR / 'faanf_melted.csv'

fanng.to_csv(sorted_path, index=False)
melted.to_csv(melted_path, index=False)

print('Saved:', sorted_path)
print('Saved:', melted_path)


Saved: /content/MSSP607/data/faanf_sorted.csv
Saved: /content/MSSP607/data/faanf_melted.csv


## Part 11. 最终检查

### 这一部分在做什么？

提交前快速检查：

* 合并后的数据是不是 1,255 行。
* `date` 是否已经是 datetime。
* `volume` 是否已经是 integer。
* `faanf.csv` 是否真的已经保存。
* melt 后是不是 6,275 行。
* 是否包含 5 个 ticker。


In [19]:
assert len(faanf) == 1255, '合并后的行数不是 1255，请检查输入文件。'
assert pd.api.types.is_datetime64_any_dtype(fanng['date']), 'date 不是 datetime。'
assert pd.api.types.is_integer_dtype(fanng['volume']), 'volume 不是 integer。'
assert faanf_path.exists(), 'faanf.csv 没有成功保存。'
assert len(melted) == 6275, 'melt 后行数不是 6275。'
assert set(fanng['ticker'].unique()) == {'AAPL', 'AMZN', 'FB', 'GOOG', 'NFLX'}, 'ticker 不完整。'

print('✅ 所有核心检查都通过。')
print('Combined rows:', len(faanf))
print('Tickers:', sorted(fanng['ticker'].unique()))
print('Melted rows:', len(melted))
print('faanf.csv exists:', faanf_path.exists())


✅ 所有核心检查都通过。
Combined rows: 1255
Tickers: ['AAPL', 'AMZN', 'FB', 'GOOG', 'NFLX']
Melted rows: 6275
faanf.csv exists: True


## Part 12. 查看 GitHub 中有哪些文件发生了变化

下面的命令**不会上传文件**，它只是查看当前 repository 相比 GitHub 版本有哪些新增或修改。

如果你刚刚生成了 `faanf.csv`、`fanng.csv`、`faanf_sorted.csv` 或 `faanf_melted.csv`，这里应该可以看到它们。


In [20]:
status = subprocess.run(
    ['git', 'status', '--short'],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True
)

print(status.stdout if status.stdout else '当前没有未保存到 Git 的文件变化。')


?? data/faanf_melted.csv
?? data/faanf_sorted.csv



## Part 13. 保存 Notebook 到 GitHub

Notebook 完成后，推荐使用 Colab 菜单：

**File → Save a copy in GitHub**

然后选择：

* 你的 `MSSP6070` repository
* Week 5 对应的文件夹
* 文件名建议：`Week_05_Participation_Activity.ipynb`

Commit message 可以写：

```text
Complete Week 5 participation activity
```

保存后进入 GitHub 网站检查 Notebook 是否真的可以打开。

### 数据文件也要确认在 GitHub 中

本 Notebook 在 Colab 中生成 `faanf.csv` 并不会自动把它 commit 到 GitHub 远程仓库。

你可以通过 GitHub 网页上传最终 CSV，或者使用 Git 完成 add / commit / push。如果你不熟悉 Git 命令，使用 GitHub 网页上传会更简单。


## Part 14. 最终 Canvas 提交 Checklist

提交前逐项确认：

- [ ] Notebook 已经 **Runtime → Run all**，从头到尾没有红色报错。
- [ ] Notebook 中保留了代码输出，而不是只有代码。
- [ ] `aapl.csv`, `amzn.csv`, `fb.csv`, `goog.csv`, `nflx.csv` 在 GitHub data folder 中。
- [ ] `faanf.csv` 已经生成并保存。
- [ ] Notebook 已保存到 MSSP6070 GitHub repository。
- [ ] Instructor 和 TAs 已添加为 repository collaborators。
- [ ] Canvas 的 Participation Activity Week 5 已上传最终 Notebook 和要求的数据文件。
- [ ] Notebook 里没有真实 Token，也没有打印 Token。
